# Entrenamiento de YOLOv8 (Dataset COCO -> YOLO -> Entrenamiento)
Este notebook extrae tu dataset original, convierte el formato de segmentación `.json` (COCO) al formato `.txt` de cajas de YOLO filtrando solo tus 5 clases de interés, y entrena un modelo YOLOv8.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Paso 1: Descomprimir el dataset original

In [ ]:
# Descomprimimos el archive.zip
!unzip -q "/content/drive/MyDrive/archive.zip" -d "/content/dataset_original"

### Paso 2: Filtrar a 5 clases y convertir a formato YOLO
YOLO necesita que las cajas delimitadoras estén normalizadas (de 0 a 1) en archivos `.txt`.

In [ ]:
import json
import os
import shutil
import random
from pathlib import Path

# Rutas
DATA_DIR = Path('/content/dataset_original/data') 
if not DATA_DIR.exists():
    DATA_DIR = Path('/content/dataset_original') # Por si se descomprimió sin la carpeta interna

ANN_FILE = DATA_DIR / 'annotations.json'

# Carpetas destino YOLO
YOLO_DIR = Path('/content/yolo_dataset')
os.makedirs(YOLO_DIR / 'images' / 'train', exist_ok=True)
os.makedirs(YOLO_DIR / 'images' / 'val', exist_ok=True)
os.makedirs(YOLO_DIR / 'labels' / 'train', exist_ok=True)
os.makedirs(YOLO_DIR / 'labels' / 'val', exist_ok=True)

with open(ANN_FILE, 'r', encoding='utf-8') as f:
    coco_data = json.load(f)

# Mapeo de IDs originales a los nuevos IDs de YOLO (0 al 4)
CLASS_MAPPING = {
    4: 0, 5: 0, # Botella_Plastico
    10: 1, 11: 1, 12: 1, # Lata
    20: 2, 21: 2, 22: 2, 23: 2, 24: 2, # Vaso
    13: 3, 14: 3, 15: 3, 16: 3, 17: 3, 18: 3, 19: 3, # Carton
    36: 4, 37: 4, 38: 4, 39: 4, 40: 4, 41: 4, 42: 4 # Envoltura
}

# Primero agrupamos las anotaciones por imagen
annotations_by_image = {}
for ann in coco_data['annotations']:
    if ann['category_id'] in CLASS_MAPPING:
        img_id = ann['image_id']
        if img_id not in annotations_by_image:
            annotations_by_image[img_id] = []
        annotations_by_image[img_id].append(ann)

# Extraemos info de las imágenes útiles
valid_images = []
img_info_dict = {img['id']: img for img in coco_data['images']}

for img_id in annotations_by_image.keys():
    if img_id in img_info_dict:
        valid_images.append(img_info_dict[img_id])

# División 80% train, 20% val
random.seed(42)
random.shuffle(valid_images)
split_idx = int(len(valid_images) * 0.8)
train_images = valid_images[:split_idx]
val_images = valid_images[split_idx:]

def process_split(images, split_name):
    for img in images:
        old_file_path = DATA_DIR / img['file_name']
        if not old_file_path.exists():
            continue
            
        new_file_name = str(img['file_name']).replace('/', '_').replace('\\', '_')
        # Copiar imagen
        shutil.copy2(old_file_path, YOLO_DIR / 'images' / split_name / new_file_name)
        
        # Crear archivo .txt
        txt_name = os.path.splitext(new_file_name)[0] + '.txt'
        txt_path = YOLO_DIR / 'labels' / split_name / txt_name
        
        img_w = img['width']
        img_h = img['height']
        
        with open(txt_path, 'w') as f:
            for ann in annotations_by_image[img['id']]:
                yolo_class = CLASS_MAPPING[ann['category_id']]
                # COCO bbox: [x_min, y_min, width, height]
                x_min, y_min, w, h = ann['bbox']
                
                # Normalizar para YOLO: [x_center, y_center, w, h]
                x_center = (x_min + w / 2) / img_w
                y_center = (y_min + h / 2) / img_h
                w_norm = w / img_w
                h_norm = h / img_h
                
                # Evitar salir de límites por pequeños errores de redondeo
                x_center = max(0, min(1, x_center))
                y_center = max(0, min(1, y_center))
                w_norm = max(0, min(1, w_norm))
                h_norm = max(0, min(1, h_norm))
                
                f.write(f"{yolo_class} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")

process_split(train_images, 'train')
process_split(val_images, 'val')

print(f"Conversión a YOLO completada:\n- Train: {len(train_images)} imágenes\n- Val: {len(val_images)} imágenes")

### Paso 3: Crear el archivo de configuración `dataset.yaml`

In [ ]:
yaml_content = """
path: /content/yolo_dataset
train: images/train
val: images/val

nc: 5
names: ['botella_plastico', 'lata', 'vaso', 'carton', 'envoltura']
"""
with open('/content/yolo_dataset/dataset.yaml', 'w') as f:
    f.write(yaml_content.strip())
print("dataset.yaml creado.")

### Paso 4: Instalar Ultralytics e Iniciar el Entrenamiento
YOLOv8 hace todo automáticamente a partir del `dataset.yaml`.

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

# Cargamos el modelo YOLOv8 Nano (más rápido y excelente para tiempo real)
model = YOLO("yolov8n.pt") 

# Entrenamos el modelo. YOLO te arrojará Precisión, Recall y mAP50 directamente en sus tablas al finalizar.
results = model.train(data="/content/yolo_dataset/dataset.yaml", epochs=50, imgsz=640, batch=16, device=0)

### Paso 5: Guardar el modelo YOLO en tu Google Drive

In [ ]:
# YOLO guarda sus resultados en runs/detect/train/weights/best.pt
!cp /content/runs/detect/train/weights/best.pt "/content/drive/MyDrive/yolov8_5clases.pt"
print("¡Modelo de YOLO guardado en tu Drive como yolov8_5clases.pt!")